# Python Data Types for Health Informatics

This notebook focuses on **practical Python data types** you’ll use constantly in health informatics: representing patients, encounters, vitals, labs, diagnoses, timestamps, and exchanging data (CSV/JSON/FHIR-like structures).

**You will learn how to:**
- Pick the right Python type (and Pandas dtype) for clinical and operational data
- Avoid common pitfalls (missing values, timestamps, numeric parsing)
- Validate and convert messy real-world fields (e.g., BP strings, ICD codes)
- Prepare data for analytics / ML


## 1) Quick tour: core Python data types

Python’s built-in types are the foundation:
- `int`, `float` for numeric measures
- `str` for text (IDs, codes, names)
- `bool` for flags (smoker yes/no)
- `None` for missing/unknown values
- `list`, `tuple`, `set`, `dict` for collections


In [1]:
# Core built-in types
age = 57                      # int
a1c = 6.9                     # float
icd10 = "I10"                 # str (Hypertension)
is_smoker = False             # bool
middle_name = None            # NoneType

# Collections common in health informatics
problems = ["I10", "E11.9"]   # list of ICD-10 codes
allergies = {"penicillin"}    # set of unique allergy agents
bp_reading = (120, 78)        # tuple (systolic, diastolic)
patient = {                   # dict: structured record
    "patient_id": "P00042",
    "age": age,
    "sex": "F",
    "problems": problems,
    "smoker": is_smoker,
    "a1c": a1c,
    "middle_name": middle_name
}

type(age), type(a1c), type(icd10), type(is_smoker), type(middle_name), type(patient)


(int, float, str, bool, NoneType, dict)

### Why it matters in health data
- IDs (MRN, encounter IDs) look numeric but should usually be stored as **strings** (to preserve leading zeros).
- Flags and categories should be **booleans** or **categoricals**.
- Dates/times should be `datetime` types—not strings—so you can compute intervals accurately.


## 2) Timestamps: `datetime`, `date`, `time`, and time zones

Clinical data is time-heavy: admission/discharge, medication administrations, lab collection times, device readings, etc.

**Goal:** store timestamps in a way that supports reliable calculations (length of stay, time-to-event, etc.).


In [2]:
from datetime import datetime, date, time, timezone, timedelta

admit_ts = datetime(2026, 3, 1, 14, 30)                # naive datetime (no timezone)
discharge_ts = datetime(2026, 3, 3, 10, 15)

length_of_stay = discharge_ts - admit_ts
length_of_stay


datetime.timedelta(days=1, seconds=71100)

In [3]:
# Timezone-aware datetime (recommended when data crosses sites/regions)
central = timezone(timedelta(hours=-6))  # Fixed offset example; for real work consider zoneinfo
admit_tz = datetime(2026, 3, 1, 14, 30, tzinfo=central)
admit_tz


datetime.datetime(2026, 3, 1, 14, 30, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=64800)))

## 3) Health data in tabular form: Pandas dtypes

In health informatics, you’ll often load data into **Pandas** for cleaning and analytics. Pandas has its own dtype system.

We’ll build a small synthetic dataset representing:
- patient demographics
- vitals
- labs
- diagnosis codes
- encounter timestamps


In [4]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "patient_id": ["000042", "000043", "000044", "000045"],
    "sex": ["F", "M", "F", "M"],
    "age": [57, 63, 45, 70],
    "smoker": [False, True, None, False],          # missing value present
    "sbp": [120, 142, "118", 135],                 # one string numeric
    "dbp": [78, 90, 76, "82"],                     # one string numeric
    "a1c": [6.9, 8.2, np.nan, 7.1],                # missing numeric
    "icd10_primary": ["I10", "E11.9", "I25.10", "I50.9"],
    "admit_time": ["2026-03-01 14:30", "2026-02-20 09:10", "2026-03-02 18:05", "2026-02-28 23:50"]
})

df


,patient_id,sex,age,smoker,sbp,dbp,a1c,icd10_primary,admit_time
0,000042,F,57,False,120,78,6.9,I10,2026-03-01 14:30
1,000043,M,63,True,142,90,8.2,E11.9,2026-02-20 09:10
2,000044,F,45,None,118,76,NaN,I25.10,2026-03-02 18:05
3,000045,M,70,False,135,82,7.1,I50.9,2026-02-28 23:50


In [5]:
df.dtypes


patient_id           str
sex                  str
age                int64
smoker            object
sbp               object
dbp               object
a1c              float64
icd10_primary        str
admit_time           str
dtype: object

### Common dtype issues you’ll see
- Numeric columns arrive as `object` because of mixed types (`"118"` mixed with `120`).
- Missing values can force unexpected types (e.g., booleans become `object`).
- Dates arrive as strings.


## 4) Converting and validating types safely

### 4.1 Numeric conversion with `pd.to_numeric`
Use `errors='coerce'` to turn bad values into `NaN` (instead of crashing).


In [6]:
for col in ["sbp", "dbp", "a1c"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["sbp", "dbp", "a1c"]].dtypes, df


(sbp      int64
 dbp      int64
 a1c    float64
 dtype: object,
   patient_id sex  age smoker  sbp  dbp  a1c icd10_primary        admit_time
 0     000042   F   57  False  120   78  6.9           I10  2026-03-01 14:30
 1     000043   M   63   True  142   90  8.2         E11.9  2026-02-20 09:10
 2     000044   F   45   None  118   76  NaN        I25.10  2026-03-02 18:05
 3     000045   M   70  False  135   82  7.1         I50.9  2026-02-28 23:50)

### 4.2 Datetime conversion with `pd.to_datetime`
This unlocks time-based computations.


In [7]:
df["admit_time"] = pd.to_datetime(df["admit_time"], errors="coerce")
df.dtypes


patient_id                  str
sex                         str
age                       int64
smoker                   object
sbp                       int64
dbp                       int64
a1c                     float64
icd10_primary               str
admit_time       datetime64[us]
dtype: object

In [8]:
# Example: time since admission relative to a reference timestamp
reference = pd.Timestamp("2026-03-05 12:00")
df["hours_since_admit"] = (reference - df["admit_time"]) / pd.Timedelta(hours=1)
df[["patient_id", "admit_time", "hours_since_admit"]]


,patient_id,admit_time,hours_since_admit
0,000042,2026-03-01 14:30:00,93.500000
1,000043,2026-02-20 09:10:00,314.833333
2,000044,2026-03-02 18:05:00,65.916667
3,000045,2026-02-28 23:50:00,108.166667


### 4.3 Booleans with missing values: Pandas nullable Boolean dtype
If you want a boolean column that supports missing values, use Pandas' **nullable** boolean dtype: `boolean`.


In [9]:
df["smoker"] = df["smoker"].astype("boolean")
df[["smoker"]].dtypes, df["smoker"]


(smoker    boolean
 dtype: object,
 0    False
 1     True
 2     <NA>
 3    False
 Name: smoker, dtype: boolean)

## 5) Categories and codes: use `category` for performance and clarity

Health datasets often contain repeated codes (sex, race/ethnicity, facility, ICD-10, CPT, LOINC).

Using the `category` dtype can:
- reduce memory usage
- speed up groupby operations
- make modeling pipelines cleaner


In [10]:
df["sex"] = df["sex"].astype("category")
df["icd10_primary"] = df["icd10_primary"].astype("category")

df.dtypes


patient_id                      str
sex                        category
age                           int64
smoker                      boolean
sbp                           int64
dbp                           int64
a1c                         float64
icd10_primary              category
admit_time           datetime64[us]
hours_since_admit           float64
dtype: object

In [11]:
df.memory_usage(deep=True)


Index                132
patient_id           220
sex                  104
age                   32
smoker                 8
sbp                   32
dbp                   32
a1c                   32
icd10_primary        219
admit_time            32
hours_since_admit     32
dtype: int64

## 6) Missing data: `None` vs `NaN` vs `pd.NA`

In health informatics you’ll constantly manage missingness (unknown smoking status, missing labs, etc.).

- `None`: Python missing (often in object columns)
- `NaN`: floating-point missing (NumPy)
- `pd.NA`: Pandas scalar missing (works across nullable dtypes)


In [12]:
df.isna().sum()


patient_id           0
sex                  0
age                  0
smoker               1
sbp                  0
dbp                  0
a1c                  1
icd10_primary        0
admit_time           0
hours_since_admit    0
dtype: int64

## 7) Structured clinical data: dictionaries and JSON

Many health systems exchange structured records as **JSON** (including FHIR-like structures). In Python, JSON maps naturally to:
- `dict` (objects)
- `list` (arrays)
- `str`, `int`, `float`, `bool`, `None`


In [13]:
import json

fhir_like_observation = {
    "resourceType": "Observation",
    "id": "obs-001",
    "status": "final",
    "code": {"coding": [{"system": "http://loinc.org", "code": "8480-6", "display": "Systolic blood pressure"}]},
    "subject": {"reference": "Patient/000042"},
    "effectiveDateTime": "2026-03-05T08:15:00-06:00",
    "valueQuantity": {"value": 120, "unit": "mmHg"}
}

json_str = json.dumps(fhir_like_observation, indent=2)
print(json_str[:350] + "\n...")


{
  "resourceType": "Observation",
  "id": "obs-001",
  "status": "final",
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "8480-6",
        "display": "Systolic blood pressure"
      }
    ]
  },
  "subject": {
    "reference": "Patient/000042"
  },
  "effectiveDateTime": "2026-03-05T08:15:00-06:00",
  "va
...


### Parsing JSON into Python objects


In [14]:
parsed = json.loads(json_str)
parsed["code"]["coding"][0]["code"], type(parsed)


('8480-6', dict)

## 8) Example: parsing blood pressure strings into numeric columns

A very common real-world cleanup task is converting `"120/78"` into numeric `sbp` and `dbp`.


In [15]:
import re

raw_bp = pd.Series(["120/78", "142/90", "118/76", None, "135/82", "bad_value"])

def parse_bp(bp):
    if bp is None or (isinstance(bp, float) and np.isnan(bp)):
        return (pd.NA, pd.NA)
    m = re.fullmatch(r"\s*(\d{2,3})\s*/\s*(\d{2,3})\s*", str(bp))
    if not m:
        return (pd.NA, pd.NA)
    return (int(m.group(1)), int(m.group(2)))

parsed_bp = raw_bp.apply(parse_bp)
bp_df = pd.DataFrame(parsed_bp.tolist(), columns=["sbp", "dbp"]).astype("Int64")  # nullable integer
pd.concat([raw_bp.rename("raw_bp"), bp_df], axis=1)


,raw_bp,sbp,dbp
0,120/78,120,78
1,142/90,142,90
2,118/76,118,76
3,NaN,<NA>,<NA>
4,135/82,135,82
5,bad_value,<NA>,<NA>


## 9) Choosing dtypes for common health informatics fields

**Recommended starting points:**
- `patient_id`, `encounter_id`, `mrn`: **string** (not int)
- `age`, `sbp`, `dbp`, `heart_rate`: `Int64` (nullable integer) when missing values exist
- `a1c`, `cholesterol`, `bmi`: `float64`
- `sex`, `race`, `icd10`, `cpt`, `loinc`, `facility`: `category` (especially if repeated)
- timestamps: `datetime64[ns]` (optionally timezone-aware in production pipelines)
- flags with missingness: `boolean` (nullable)


## 10) Mini exercise (optional)
1. Add a `bmi` column with some missing values.
2. Convert it to numeric.
3. Bin BMI into categories (e.g., normal/overweight/obesity) using `pd.cut` and set it to `category`.


In [16]:
# Starter cell for the exercise
df_ex = df.copy()

df_ex["bmi"] = ["27.1", 31.8, None, 24.9]
df_ex["bmi"] = pd.to_numeric(df_ex["bmi"], errors="coerce")

df_ex["bmi_group"] = pd.cut(
    df_ex["bmi"],
    bins=[0, 25, 30, 100],
    labels=["normal", "overweight", "obesity"],
    right=False
).astype("category")

df_ex[["patient_id", "bmi", "bmi_group"]], df_ex.dtypes


(  patient_id   bmi   bmi_group
 0     000042  27.1  overweight
 1     000043  31.8     obesity
 2     000044   NaN         NaN
 3     000045  24.9      normal,
 patient_id                      str
 sex                        category
 age                           int64
 smoker                      boolean
 sbp                           int64
 dbp                           int64
 a1c                         float64
 icd10_primary              category
 admit_time           datetime64[us]
 hours_since_admit           float64
 bmi                         float64
 bmi_group                  category
 dtype: object)

---
## Summary
You now have a practical toolkit for choosing and converting data types in Python for health informatics:
- Use the right **built-in types** for structured records.
- In Pandas, convert early: `to_numeric`, `to_datetime`, nullable `Int64` and `boolean`.
- Use `category` for repeated codes.
- Treat timestamps carefully for time-based analyses.
